# 11 하이브리드 수요예측

**type별 SBC(rule-base) vs ML 클러스터링** — 제품수(검증 판매량) 반영 **가중 WMAPE** 비교

> 논문: 변동성 큰 **B센터**에서는 SBC가, 저변동 **A센터**에서는 ML이 유리했음.  
> 본 데이터는 type별 변동성 차이가 상대적으로 작아 **결과가 항상 같지 않으며**, 센터·데이터 특성에 따라 우세 scheme이 달라질 수 있음.

### ⓪ 환경 설정

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.experiment_data import load_forecast_frames

df, feat_df = load_forecast_frames()
phase2 = pd.read_parquet(DATA_PROCESSED / 'phase2_results.parquet')
phase2_best = pd.read_csv(DATA_PROCESSED / 'phase2_best_per_condition.csv')
print('Phase2 (XGBoost×6임베딩) | rows:', len(phase2), '| 조건:', len(phase2_best))


Phase2 (XGBoost×6임베딩) | rows: 1980 | 조건: 35


### ① 가중 WMAPE — type별 SBC vs ML

In [2]:
import numpy as np
from utils.phase_analysis import (
    validation_weights, build_family_from_phase2_best,
    compare_schemes_by_type,
)

val_weights = validation_weights(df)
# 10장: 조건별 XGBoost+Best 임베딩 예측으로 type별 scheme 비교
family_final = build_family_from_phase2_best(phase2, phase2_best, val_weights)

type_compare = compare_schemes_by_type(family_final)
display(type_compare)
print('가중 WMAPE 우세:', type_compare['better_scheme_weighted'].value_counts().to_dict())

cv = df[df['yearweek'] <= TRAIN_WEEK_MAX].groupby('type')['sales'].agg(['mean', 'std'])
cv['cv'] = (cv['std'] / cv['mean'].replace(0, np.nan)).round(3)
print('\n=== type별 학습구간 판매 변동계수(CV) ===')
print(cv)


,n_products,SBC_wmape_weighted,ML_wmape_weighted,SBC_wmape_mean,ML_wmape_mean,delta_weighted_SBC_minus_ML,better_scheme_weighted
A,33,33.90,35.93,90.12,111.41,-2.03,SBC
B,33,27.07,28.55,92.10,51.23,-1.48,SBC
C,33,33.33,32.89,55.57,74.81,0.44,ML
D,33,37.78,37.44,97.20,65.20,0.34,ML
E,33,27.89,28.92,37.12,38.81,-1.03,SBC


가중 WMAPE 우세: {'SBC': 3, 'ML': 2}

=== type별 학습구간 판매 변동계수(CV) ===
              mean            std     cv
type                                    
A     44216.910475  100342.898923  2.269
B     18162.835311   45927.000022  2.529
C     20579.720066   53018.856163  2.576
D     44015.913722   99010.064531  2.249
E      7464.302488   19194.165941  2.571


### 해석

- **가중 WMAPE** = Σ(WMAPE_f × 검증판매량_f) / Σ(검증판매량_f)
- 논문과 달리 본 실험에서는 type 간 변동성 격차가 크지 않을 수 있음 → scheme 우세가 센터마다 다르게 나타남
- `final_best_per_condition.csv` → 12장 RIDR 분석 이후 활용